# 03 · TTC Schedule Analysis (Static GTFS)

Using the TTC's published timetable, this notebook calculates:

1. The **scheduled headway** of every route in each time period
2. Which routes belong to the **frequent network**
3. The **500 m walking catchment** of frequent stops, and which stops fall outside it (service gaps)

The results also serve as the baseline for the realtime analysis: once enough realtime data is collected, it is compared against this schedule ("planned vs. actual").

## 1. Settings

Change the date, the frequent-service threshold or the walking distance here.

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

GTFS = Path("../data/static")          # folder with the unzipped static GTFS
OUT = Path("../data/processed")        # outputs (inside data/, which is git-ignored)
OUT.mkdir(parents=True, exist_ok=True)

SERVICE_DATE = "20261021"   # a typical weekday (Wednesday); must fall within calendar.txt validity
FREQUENT_MIN = 10           # headway <= 10 minutes = frequent
WALK_M = 500                # walking distance in metres
CRS_M = "EPSG:2952"         # NAD83(CSRS) / MTM zone 10: metric projection commonly used for Toronto

# Time periods in hours; values above 24 are after midnight on the same service day (GTFS convention)
PERIODS = {
    "AM peak": (6, 9),
    "Midday": (9, 15),
    "PM peak": (15, 19),
    "Evening": (19, 22),
    "Late evening": (22, 25),
}
CORE_PERIODS = ["AM peak", "Midday", "PM peak"]   # all three must meet the threshold to count as frequent

## 2. Load GTFS

In [ ]:
def read(name):
    path = GTFS / f"{name}.txt"
    return pd.read_csv(path, dtype=str) if path.exists() else pd.DataFrame()

routes = read("routes")
trips = read("trips")
stop_times = read("stop_times")
stops = read("stops")
cal = read("calendar")
cal_dates = read("calendar_dates")

# direction_id is optional in GTFS; default to "0" so later steps still work
if "direction_id" not in trips.columns:
    trips["direction_id"] = "0"
trips["direction_id"] = trips["direction_id"].fillna("0")

for name, df in [("routes", routes), ("trips", trips), ("stop_times", stop_times), ("stops", stops)]:
    print(f"{name:11s} {len(df):>10,} rows")
if len(cal):
    print("Timetable valid from", cal["start_date"].min(), "to", cal["end_date"].max())

## 3. Select the trips that run on the chosen date

`calendar.txt` defines which days of the week and date range each `service_id` runs; `calendar_dates.txt` holds exceptions such as holidays (added or removed service).

In [ ]:
def active_services(date):
    weekday = pd.Timestamp(date).day_name().lower()   # e.g. "wednesday"
    active = set()
    if len(cal):
        ok = (cal["start_date"] <= date) & (cal["end_date"] >= date) & (cal[weekday] == "1")
        active |= set(cal.loc[ok, "service_id"])
    if len(cal_dates):
        cd = cal_dates[cal_dates["date"] == date]
        active |= set(cd.loc[cd["exception_type"] == "1", "service_id"])   # service added
        active -= set(cd.loc[cd["exception_type"] == "2", "service_id"])   # service removed
    return active

services = active_services(SERVICE_DATE)
day_trips = trips[trips["service_id"].isin(services)]
print(f"{SERVICE_DATE}: {len(services)} service_id(s), {len(day_trips):,} trips, {day_trips['route_id'].nunique()} routes")
assert len(day_trips) > 0, "No trips on this date: check that SERVICE_DATE is within the timetable's validity"


## 4. Departure time of each trip

Each trip is assigned to a time period by its departure time from its **first stop**. GTFS times can exceed 24:00 (e.g. `25:10:00` = 1:10 am the next morning, still on this service day), so they are converted to minutes after midnight.

In [ ]:
def to_minutes(s):
    p = s.str.split(":", expand=True).astype(int)
    return p[0] * 60 + p[1] + p[2] / 60

st = stop_times[stop_times["trip_id"].isin(day_trips["trip_id"])].copy()
st["stop_sequence"] = st["stop_sequence"].astype(int)

first = (
    st.dropna(subset=["departure_time"])
      .sort_values("stop_sequence")
      .groupby("trip_id", as_index=False)
      .first()
)
first["dep_min"] = to_minutes(first["departure_time"])
first = first.merge(day_trips[["trip_id", "route_id", "direction_id"]], on="trip_id")

def period_of(minutes):
    h = minutes / 60
    for name, (a, b) in PERIODS.items():
        if a <= h < b:
            return name
    return None

first["period"] = first["dep_min"].apply(period_of)
first[["trip_id", "route_id", "direction_id", "departure_time", "period"]].head()

### Sanity checks

- **Late-night time format**: agencies write after-midnight trips either as `25:30` (same service day) or `01:30` (next service day). Both are valid GTFS.
- **Duplicates**: overlapping service_ids on the same day would double-count trips and make headways look too short.
- **Branches / short turns**: a route with several starting stops is a trunk with branches, so its combined headway is shorter than any single branch.

In [ ]:
print("Earliest departure:", first["departure_time"].min())
print("Latest departure:  ", first["departure_time"].max())
print("Trips starting at/after 24:00:", (first["dep_min"] >= 24 * 60).sum())
print("Trips starting 00:00-04:00:   ", (first["dep_min"] < 4 * 60).sum())

print("\nActive service_ids:", sorted(services))
dup = first.duplicated(subset=["route_id", "direction_id", "stop_id", "departure_time"], keep=False)
print("Possible duplicate trips:", dup.sum())

# Starting stops per route at midday (several = branches or short turns)
for r in ["63", "25", "47"]:
    match = routes.loc[routes["route_short_name"] == r, "route_id"]
    if match.empty:
        continue
    sub = first[(first["route_id"] == match.iloc[0]) & (first["period"] == "Midday")]
    print(f"\nRoute {r}: midday trips by starting stop")
    print(sub.merge(stops[["stop_id", "stop_name"]], on="stop_id")
             .groupby(["direction_id", "stop_name"]).size())

## 5. Scheduled headway

Headway = period length / number of trips departing in that period. Each route has two directions; the **less frequent direction** is used, so a route only counts as frequent if both directions are (a conservative choice).

In [ ]:
counts = (
    first.dropna(subset=["period"])
         .groupby(["route_id", "direction_id", "period"])
         .size().rename("trips").reset_index()
)
counts["period_min"] = counts["period"].map(lambda p: (PERIODS[p][1] - PERIODS[p][0]) * 60)
counts["headway_min"] = counts["period_min"] / counts["trips"]

hw = (
    counts.groupby(["route_id", "period"])["headway_min"].max()   # worse of the two directions
          .unstack("period")
          .reindex(columns=list(PERIODS))
)
hw = (
    routes.set_index("route_id")[["route_short_name", "route_long_name"]]
          .join(hw, how="right")
          .round(1)
)
hw["frequent"] = hw[CORE_PERIODS].le(FREQUENT_MIN).all(axis=1)   # no service (NaN) = not frequent

print(f"Frequent routes ({'/'.join(CORE_PERIODS)} all <= {FREQUENT_MIN} min): {hw['frequent'].sum()} of {len(hw)}")
hw.sort_values("Midday").head(20)

In [ ]:
hw.to_csv(OUT / f"scheduled_headway_{SERVICE_DATE}.csv")

ax = hw["Midday"].dropna().plot.hist(bins=range(0, 65, 5), figsize=(8, 4), edgecolor="white")
ax.axvline(FREQUENT_MIN, color="black", linestyle="--", linewidth=1)
ax.text(FREQUENT_MIN + 0.5, ax.get_ylim()[1] * 0.9, f"{FREQUENT_MIN} min", fontsize=9)
ax.set_xlabel("Scheduled midday headway (minutes)")
ax.set_ylabel("Number of routes")
ax.set_title(f"TTC midday headway distribution ({SERVICE_DATE})")
plt.tight_layout()
plt.savefig(OUT / "midday_headway_hist.png", dpi=150)
plt.show()

## 6. Flag frequent stops

A stop is frequent if **any** frequent route serves it. Coordinates are reprojected from latitude/longitude (EPSG:4326) to the metric EPSG:2952 so a 500 m buffer is accurate.

In [ ]:
stop_routes = (
    st[["trip_id", "stop_id"]].drop_duplicates()
      .merge(day_trips[["trip_id", "route_id"]], on="trip_id")[["stop_id", "route_id"]]
      .drop_duplicates()
)
freq_routes = set(hw.index[hw["frequent"]])
is_freq_stop = stop_routes.assign(f=stop_routes["route_id"].isin(freq_routes)).groupby("stop_id")["f"].any()

served = stops["stop_id"].isin(is_freq_stop.index).to_numpy()
gstops = gpd.GeoDataFrame(
    stops[served].copy(),
    geometry=gpd.points_from_xy(stops["stop_lon"].astype(float), stops["stop_lat"].astype(float))[served],
    crs="EPSG:4326",
).to_crs(CRS_M)
gstops["frequent"] = gstops["stop_id"].map(is_freq_stop)
print(f"Stops served on this date: {len(gstops):,}, of which frequent: {gstops['frequent'].sum():,}")

## 7. 500 m catchment

Buffer each frequent stop by 500 m and dissolve the buffers into one area. Stops outside that area have no frequent service nearby and are **service gap** candidates.

In [ ]:
freq_area = gstops.loc[gstops["frequent"]].buffer(WALK_M).union_all()
gstops["near_frequent"] = gstops.within(freq_area)

share = gstops["near_frequent"].mean()
gaps = gstops[~gstops["near_frequent"]]
print(f"{share:.1%} of stops are within {WALK_M} m of a frequent stop; {len(gaps):,} stops are outside")
print(f"Frequent catchment area: {freq_area.area / 1e6:,.1f} km²")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
gpd.GeoSeries([freq_area], crs=CRS_M).plot(ax=ax, color="#4C78A8", alpha=0.25)
gstops[gstops["frequent"]].plot(ax=ax, color="#4C78A8", markersize=2, label="Frequent stop")
gaps.plot(ax=ax, color="#E45756", markersize=4, label="Outside catchment")
ax.set_title(f"TTC frequent network {WALK_M} m catchment ({SERVICE_DATE}, headway <= {FREQUENT_MIN} min)")
ax.set_axis_off()
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(OUT / "frequent_coverage_map.png", dpi=150)
plt.show()

Export GeoJSON for further mapping in QGIS, ArcGIS or Power BI.

In [ ]:
gpd.GeoDataFrame(geometry=[freq_area], crs=CRS_M).to_crs("EPSG:4326").to_file(OUT / "frequent_coverage.geojson", driver="GeoJSON")
gstops.to_crs("EPSG:4326")[["stop_id", "stop_name", "frequent", "near_frequent", "geometry"]].to_file(
    OUT / "stops_coverage.geojson", driver="GeoJSON"
)
print("Written:", *sorted(p.name for p in OUT.iterdir()), sep="\n  ")

## 8. (Optional) Population coverage

To estimate how many residents live within 500 m of a frequent stop, you need Statistics Canada dissemination area (DA) boundaries and population counts (free to download).

Join the two, save them as one file with a population column, then set `DA_FILE` and `POP_COL`. DA centroids are used to decide coverage: simple, but less accurate where DAs are large.

In [ ]:
DA_FILE = None          # e.g. Path("../data/census/toronto_da_pop.gpkg")
POP_COL = "population"

if DA_FILE and Path(DA_FILE).exists():
    da = gpd.read_file(DA_FILE).to_crs(CRS_M)
    da["covered"] = da.geometry.centroid.within(freq_area)
    covered = da.loc[da["covered"], POP_COL].sum()
    total = da[POP_COL].sum()
    print(f"About {covered:,.0f} of {total:,.0f} residents ({covered / total:.1%}) live within {WALK_M} m of a frequent stop (DA centroid method)")
else:
    print("DA_FILE not set; skipping population analysis")

## 9. Limitations (state these in the README)

- **Straight-line distance**: the 500 m buffer is as the crow flies; real walking routes are longer. Next step: network-based walksheds with OSMnx and OpenStreetMap
- **Trips counted at the first stop**: branches and short turns can make a route's combined headway shorter than what riders see on a given segment
- **Route-level classification**: where several routes share a street, the combined headway is shorter than any single route. A stop-level "trips per hour, all routes" measure would capture this
- **One day only**: weekends and holidays differ; change `SERVICE_DATE` and rerun to compare
- **Frequent definition**: `FREQUENT_MIN` and `CORE_PERIODS` are assumptions and can be adjusted